# Christ University Computer Science Department Web Scraper

This notebook scrapes information from the Christ University Computer Science department page.

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime

In [9]:
def scrape_christ_university_cs_dept(url):
    """
    Scrape information from Christ University Computer Science department page
    
    Parameters:
    url (str): URL of the Christ University CS department page
    
    Returns:
    dict: Dictionary containing scraped data
    """
    # Set up Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run in headless mode
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    
    # Initialize the Chrome driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    scraped_data = {
        'page_title': '',
        'department_info': '',
        'faculty_info': [],
        'programs_offered': [],
        'research_areas': [],
        'contact_info': {},
        'news_updates': [],
        'all_text_content': '',
        'links': [],
        'images': []
    }
    
    try:
        # Open the webpage
        driver.get(url)
        print(f"Opened webpage: {url}")
        
        # Wait for the page to load
        time.sleep(5)
        
        # Get page title
        scraped_data['page_title'] = driver.title
        print(f"Page title: {scraped_data['page_title']}")
        
        # Get all text content
        body_element = driver.find_element(By.TAG_NAME, "body")
        scraped_data['all_text_content'] = body_element.text
        
        # Extract specific sections
        try:
            # Look for department description/overview
            dept_sections = driver.find_elements(By.CSS_SELECTOR, ".department-overview, .about-department, .dept-info, .overview, .description")
            if dept_sections:
                scraped_data['department_info'] = dept_sections[0].text
            else:
                # Try to find main content area
                main_content = driver.find_elements(By.CSS_SELECTOR, "main, .main-content, .content, .page-content")
                if main_content:
                    scraped_data['department_info'] = main_content[0].text[:1000] + "..."
        except Exception as e:
            print(f"Error extracting department info: {e}")
        
        # Extract faculty information
        try:
            faculty_elements = driver.find_elements(By.CSS_SELECTOR, ".faculty, .staff, .faculty-member, .teacher")
            for faculty in faculty_elements:
                faculty_data = {
                    'name': '',
                    'designation': '',
                    'details': faculty.text
                }
                
                # Try to extract name and designation
                name_elem = faculty.find_elements(By.CSS_SELECTOR, ".name, .faculty-name, h3, h4")
                if name_elem:
                    faculty_data['name'] = name_elem[0].text
                
                designation_elem = faculty.find_elements(By.CSS_SELECTOR, ".designation, .title, .position")
                if designation_elem:
                    faculty_data['designation'] = designation_elem[0].text
                
                scraped_data['faculty_info'].append(faculty_data)
        except Exception as e:
            print(f"Error extracting faculty info: {e}")
        
        # Extract programs/courses information
        try:
            program_elements = driver.find_elements(By.CSS_SELECTOR, ".program, .course, .degree, .curriculum")
            for program in program_elements:
                program_text = program.text.strip()
                if program_text and len(program_text) > 10:
                    scraped_data['programs_offered'].append(program_text)
        except Exception as e:
            print(f"Error extracting programs: {e}")
        
        # Extract research areas
        try:
            research_elements = driver.find_elements(By.CSS_SELECTOR, ".research, .research-area, .specialization")
            for research in research_elements:
                research_text = research.text.strip()
                if research_text and len(research_text) > 10:
                    scraped_data['research_areas'].append(research_text)
        except Exception as e:
            print(f"Error extracting research areas: {e}")
        
        # Extract contact information
        try:
            contact_elements = driver.find_elements(By.CSS_SELECTOR, ".contact, .contact-info, .address")
            if contact_elements:
                contact_text = contact_elements[0].text
                
                # Extract email addresses
                emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', contact_text)
                scraped_data['contact_info']['emails'] = emails
                
                # Extract phone numbers
                phones = re.findall(r'[+]?[1-9]?[0-9]{7,14}', contact_text)
                scraped_data['contact_info']['phones'] = phones
                
                scraped_data['contact_info']['full_text'] = contact_text
        except Exception as e:
            print(f"Error extracting contact info: {e}")
        
        # Extract all links
        try:
            link_elements = driver.find_elements(By.TAG_NAME, "a")
            for link in link_elements:
                href = link.get_attribute("href")
                text = link.text.strip()
                if href and text:
                    scraped_data['links'].append({
                        'url': href,
                        'text': text
                    })
        except Exception as e:
            print(f"Error extracting links: {e}")
        
        # Extract images
        try:
            img_elements = driver.find_elements(By.TAG_NAME, "img")
            for img in img_elements:
                src = img.get_attribute("src")
                alt = img.get_attribute("alt")
                if src:
                    scraped_data['images'].append({
                        'src': src,
                        'alt': alt or 'No alt text'
                    })
        except Exception as e:
            print(f"Error extracting images: {e}")
        
        # Extract news/updates if available
        try:
            news_elements = driver.find_elements(By.CSS_SELECTOR, ".news, .updates, .announcements, .events")
            for news in news_elements:
                news_text = news.text.strip()
                if news_text and len(news_text) > 20:
                    scraped_data['news_updates'].append(news_text)
        except Exception as e:
            print(f"Error extracting news: {e}")
        
        print(f"Successfully scraped data from {url}")
        print(f"Found {len(scraped_data['faculty_info'])} faculty members")
        print(f"Found {len(scraped_data['programs_offered'])} programs")
        print(f"Found {len(scraped_data['links'])} links")
        print(f"Found {len(scraped_data['images'])} images")
        
        return scraped_data
        
    except Exception as e:
        print(f"Error during scraping: {e}")
        return scraped_data
        
    finally:
        # Close the browser
        driver.quit()

In [10]:
# URL to scrape
url = "https://christuniversity.in/departments/main-campus/school-of-sciences/computer-science"

# Scrape the website
scraped_data = scrape_christ_university_cs_dept(url)

# Display basic information
print("\n=== SCRAPED DATA SUMMARY ===")
print(f"Page Title: {scraped_data['page_title']}")
print(f"\nDepartment Info (first 500 chars):\n{scraped_data['department_info'][:500]}...")
print(f"\nNumber of Faculty Members: {len(scraped_data['faculty_info'])}")
print(f"Number of Programs: {len(scraped_data['programs_offered'])}")
print(f"Number of Links: {len(scraped_data['links'])}")
print(f"Number of Images: {len(scraped_data['images'])}")

Opened webpage: https://christuniversity.in/departments/main-campus/school-of-sciences/computer-science
Page title: CHRIST UNIVERSITY
Page title: CHRIST UNIVERSITY
Successfully scraped data from https://christuniversity.in/departments/main-campus/school-of-sciences/computer-science
Found 0 faculty members
Found 0 programs
Found 33 links
Found 179 images

=== SCRAPED DATA SUMMARY ===
Page Title: CHRIST UNIVERSITY

Department Info (first 500 chars):
...

Number of Faculty Members: 0
Number of Programs: 0
Number of Links: 33
Number of Images: 179
Successfully scraped data from https://christuniversity.in/departments/main-campus/school-of-sciences/computer-science
Found 0 faculty members
Found 0 programs
Found 33 links
Found 179 images

=== SCRAPED DATA SUMMARY ===
Page Title: CHRIST UNIVERSITY

Department Info (first 500 chars):
...

Number of Faculty Members: 0
Number of Programs: 0
Number of Links: 33
Number of Images: 179


In [11]:
# Create DataFrames for different types of data

# Faculty DataFrame
if scraped_data['faculty_info']:
    faculty_df = pd.DataFrame(scraped_data['faculty_info'])
    print("Faculty Information:")
    print(faculty_df.head())
else:
    print("No faculty information found")

# Programs DataFrame
if scraped_data['programs_offered']:
    programs_df = pd.DataFrame({'Program': scraped_data['programs_offered']})
    print("\nPrograms Offered:")
    print(programs_df.head())
else:
    print("No programs information found")

# Links DataFrame
if scraped_data['links']:
    links_df = pd.DataFrame(scraped_data['links'])
    print("\nLinks (first 10):")
    print(links_df.head(10))
else:
    print("No links found")

No faculty information found
No programs information found

Links (first 10):
                                                 url  \
0  https://christuniversity.in/schools/school-of-...   
1  https://christuniversity.in/our-programmes/UGP...   
2  https://christuniversity.in/our-programmes/PGP...   
3  https://christuniversity.in/our-programmes/DP(...   
4                                 javascript:void(0)   
5                                 javascript:void(0)   
6  https://christuniversity.in/department-festiva...   
7  https://christuniversity.in/department-activit...   
8  https://christuniversity.in/department-activit...   
9  https://christuniversity.in/department-activit...   

                                                text  
0                                Back to School Page  
1  UNDERGRADUATE\nComprehensive undergraduate pro...  
2  POSTGRADUATE\nPostgraduate studies focused on ...  
3  DOCTORAL (PhD)\nDoctoral programmes fostering ...  
4                             

In [12]:
# Save the scraped data to files
import json

# Save complete data as JSON
with open('christ_university_cs_data.json', 'w', encoding='utf-8') as f:
    json.dump(scraped_data, f, indent=2, ensure_ascii=False)

# Save specific data to CSV files
if scraped_data['faculty_info']:
    faculty_df = pd.DataFrame(scraped_data['faculty_info'])
    faculty_df.to_csv('christ_university_faculty.csv', index=False)
    print("Faculty data saved to christ_university_faculty.csv")

if scraped_data['links']:
    links_df = pd.DataFrame(scraped_data['links'])
    links_df.to_csv('christ_university_links.csv', index=False)
    print("Links data saved to christ_university_links.csv")

if scraped_data['programs_offered']:
    programs_df = pd.DataFrame({'Program': scraped_data['programs_offered']})
    programs_df.to_csv('christ_university_programs.csv', index=False)
    print("Programs data saved to christ_university_programs.csv")

# Save all text content to a text file
with open('christ_university_all_text.txt', 'w', encoding='utf-8') as f:
    f.write(scraped_data['all_text_content'])
    print("All text content saved to christ_university_all_text.txt")

print("\nAll data saved successfully!")
print(f"Scraping completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Links data saved to christ_university_links.csv
All text content saved to christ_university_all_text.txt

All data saved successfully!
Scraping completed at: 2025-08-02 09:38:33


In [14]:
# Extract specific faculty information from the HTML structure
def extract_faculty_cards(url):
    """
    Extract faculty information from the specific HTML structure shown
    """
    # Set up Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    faculty_data = []
    
    try:
        driver.get(url)
        print(f"Opened webpage: {url}")
        time.sleep(5)
        
        # Look for faculty cards with the specific structure
        faculty_cards = driver.find_elements(By.CSS_SELECTOR, ".facly-items, .col-lg-3.col-sm-6.col-md-3")
        
        print(f"Found {len(faculty_cards)} faculty cards")
        
        for card in faculty_cards:
            try:
                faculty_info = {}
                
                # Extract image URL
                try:
                    img_element = card.find_element(By.CSS_SELECTOR, ".facly-mmg_123q img")
                    faculty_info['Image_URL'] = img_element.get_attribute('src')
                    faculty_info['Image_Alt'] = img_element.get_attribute('alt')
                except:
                    faculty_info['Image_URL'] = 'Not found'
                    faculty_info['Image_Alt'] = 'Not found'
                
                # Extract faculty name
                try:
                    name_element = card.find_element(By.CSS_SELECTOR, ".facly-texts_123q h3")
                    faculty_info['Name'] = name_element.text.strip()
                except:
                    faculty_info['Name'] = 'Not found'
                
                # Extract department
                try:
                    dept_element = card.find_element(By.CSS_SELECTOR, ".facly-texts_123q h4")
                    faculty_info['Department'] = dept_element.text.strip()
                except:
                    faculty_info['Department'] = 'Not found'
                
                # Extract specialization
                try:
                    spec_element = card.find_element(By.CSS_SELECTOR, ".facly-texts_123q h5")
                    faculty_info['Specialization'] = spec_element.text.strip()
                except:
                    faculty_info['Specialization'] = 'Not found'
                
                # Extract profile link
                try:
                    link_element = card.find_element(By.CSS_SELECTOR, ".vwew_bttn a")
                    faculty_info['Profile_Link'] = link_element.get_attribute('href')
                except:
                    faculty_info['Profile_Link'] = 'Not found'
                
                # Only add if we found meaningful data
                if faculty_info['Name'] != 'Not found' or faculty_info['Department'] != 'Not found':
                    faculty_data.append(faculty_info)
                    print(f"Extracted: {faculty_info['Name']} - {faculty_info['Department']}")
                
            except Exception as e:
                print(f"Error extracting faculty card: {e}")
                continue
        
        return faculty_data
        
    except Exception as e:
        print(f"Error during faculty extraction: {e}")
        return faculty_data
        
    finally:
        driver.quit()

# Extract faculty data using the specific structure
url = "https://christuniversity.in/departments/main-campus/school-of-sciences/computer-science"
faculty_cards_data = extract_faculty_cards(url)

print(f"\n=== EXTRACTED {len(faculty_cards_data)} FACULTY MEMBERS ===")

# Create DataFrame and save to CSV
if faculty_cards_data:
    faculty_cards_df = pd.DataFrame(faculty_cards_data)
    
    # Display the data
    print("\nFaculty Information:")
    print(faculty_cards_df.to_string(index=False))
    
    # Save to CSV
    faculty_cards_df.to_csv('christ_university_faculty_cards.csv', index=False)
    print(f"\nFaculty data saved to 'christ_university_faculty_cards.csv'")
    
    # Also save as Excel for better formatting
    faculty_cards_df.to_excel('christ_university_faculty_cards.xlsx', index=False)
    print(f"Faculty data also saved to 'christ_university_faculty_cards.xlsx'")
    
else:
    print("No faculty data found with the specified structure")

Opened webpage: https://christuniversity.in/departments/main-campus/school-of-sciences/computer-science
Found 24 faculty cards
Extracted: ALWIN JOSEPH - COMPUTER SCIENCE(LAVASA)
Extracted: ALWIN JOSEPH - COMPUTER SCIENCE(LAVASA)
Extracted: Dr AMRUTHA K - COMPUTER SCIENCE
Extracted: Dr AMRUTHA K - COMPUTER SCIENCE
Extracted: ANANYA MITRA - COMPUTER SCIENCE(LAVASA)
Extracted: ANANYA MITRA - COMPUTER SCIENCE(LAVASA)
Extracted: Dr ANUSHA JAMES - COMPUTER SCIENCE (YESHWANTHPUR)
Extracted: Dr ANUSHA JAMES - COMPUTER SCIENCE (YESHWANTHPUR)
Extracted: Dr AROKIA PAUL RAJAN R - COMPUTER SCIENCE(LAVASA)
Found 24 faculty cards
Extracted: ALWIN JOSEPH - COMPUTER SCIENCE(LAVASA)
Extracted: ALWIN JOSEPH - COMPUTER SCIENCE(LAVASA)
Extracted: Dr AMRUTHA K - COMPUTER SCIENCE
Extracted: Dr AMRUTHA K - COMPUTER SCIENCE
Extracted: ANANYA MITRA - COMPUTER SCIENCE(LAVASA)
Extracted: ANANYA MITRA - COMPUTER SCIENCE(LAVASA)
Extracted: Dr ANUSHA JAMES - COMPUTER SCIENCE (YESHWANTHPUR)
Extracted: Dr ANUSHA JAMES

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
# Display some sample content
print("=== SAMPLE CONTENT ===")
print(f"\nFirst 1000 characters of all text content:")
print(scraped_data['all_text_content'][:1000])

if scraped_data['contact_info']:
    print("\n=== CONTACT INFORMATION ===")
    for key, value in scraped_data['contact_info'].items():
        print(f"{key}: {value}")

if scraped_data['images']:
    print(f"\n=== FIRST 5 IMAGES ===")
    for i, img in enumerate(scraped_data['images'][:5]):
        print(f"{i+1}. {img['alt']} - {img['src'][:100]}...")

=== SAMPLE CONTENT ===

First 1000 characters of all text content:
CHRIST (Deemed to be University) | Central Campus | Hosur Road | Bangalore
MENU
Computer Science
Departments in School of Sciences
Back to School Page
SIDEBAR
UNDERGRADUATE
Comprehensive undergraduate programmes building strong academic and professional foundations.
POSTGRADUATE
Postgraduate studies focused on advanced knowledge, research, and career specialization.
DOCTORAL (PhD)
Doctoral programmes fostering original research, innovation, and academic excellence.
The Department of Computer Science of CHRIST (Deemed to be University) strives to shape outstanding computer professionals with ethical and human values to reshape the nation's destiny. The training imparted aims to prepare young minds for the challenging opportunities in the IT industry with a global awareness rooted in the Indian  
  Read more...








The Department of Computer Science endeavors to imbibe the vision of the University “Excellence and Serv